# Stage 3 — Clinical NLP: End-to-End Pipeline Experiment
**Project**: Personalized Precision Medicine for Oncology Treatment Optimization  
**Objective**: Demonstrates conservative clinical text preprocessing, rule-based negation scoping, concept extraction, feature vectorization, baseline model evaluation, and error analysis on the official `TRAIN` and `VALIDATION` partitions.  
---
> **Safety & Isolation Rule**: The locked test set (`LOCKED_TEST`) is strictly held out and will NOT be evaluated, inspected, or tuned in this notebook.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Add src to sys.path
SRC_DIR = Path("../src").resolve()
sys.path.insert(0, str(SRC_DIR))

import data_loader
import text_cleaning
import text_normalization
import sentence_processing
import negation_detection
import clinical_concepts
import feature_extraction
import label_preparation
import baseline
import inference

print("Clinical NLP modules loaded successfully!")

## 1. Load Data Partitions and Verify Schema

In [ ]:
df_train = data_loader.load_train_data()
df_val = data_loader.load_validation_data()

print(f"TRAIN shape: {df_train.shape} (Patients: {df_train['patient_id'].nunique()})")
print(f"VALIDATION shape: {df_val.shape} (Patients: {df_val['patient_id'].nunique()})")

# Display sample record (privacy-safe)
sample_row = df_train.iloc[0]
print("Sample Document ID:", sample_row["document_id"])
print("Note Type:", sample_row["document_type"])
print("Urgency:", sample_row["urgency_level"], "| Hazard:", sample_row["hazard_type"])
print("\nText Snippet:\n", sample_row["text"][:200], "...")

## 2. Conservative Text Cleaning & Normalization

In [ ]:
raw_note = sample_row["text"]
cleaned = text_cleaning.clean_clinical_text(raw_note)
normalized = text_normalization.normalize_clinical_text(cleaned)

print(f"Raw Length: {len(raw_note)} chars | Cleaned Length: {len(cleaned)} chars")
print("\nNormalized First 150 chars:\n", normalized[:150])

## 3. Sentence Boundary Detection

In [ ]:
sentences = sentence_processing.split_into_sentences(normalized)
print(f"Segmented into {len(sentences)} clinical sentences:")
for s in sentences[:3]:
    print(f"  [{s['sentence_id']}] ({s['start_char']}-{s['end_char']}): {s['text']}")

## 4. Clinical Concept Extraction & Negation Scoping

In [ ]:
extracted_entities = clinical_concepts.extract_clinical_concepts(normalized, assign_polarity=True)
print(f"Extracted {len(extracted_entities)} clinical concepts:")
pd.DataFrame(extracted_entities)

## 5. Negation-Aware Feature Vectorization

In [ ]:
scoped_text = feature_extraction.create_negation_scoped_text(normalized)
print("Sample Negation-Scoped Tokens:\n", scoped_text[:250], "...")

feature_pipe = feature_extraction.ClinicalFeaturePipeline.load()
X_val, struct_val = feature_pipe.transform(df_val.head(10))
print(f"\nCombined Feature Matrix Shape (Top 10 Val): {X_val.shape}")
struct_val.head(3)

## 6. Baseline Models Validation Performance

In [ ]:
with open("../results/metrics/baseline_validation_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("=== PRIMARY URGENCY LEVEL CLASSIFICATION (4-Class) ===")
print(f"Validation Macro F1: {metrics['urgency_classification']['macro_f1']:.4f}")
print(f"Validation Accuracy: {metrics['urgency_classification']['accuracy']:.4f}")

urg_rep = pd.DataFrame(metrics['urgency_classification']['classification_report']).T
display(urg_rep)

print("\n=== SECONDARY TOXICITY HAZARD CLASSIFICATION (8-Class) ===")
print(f"Validation Macro F1: {metrics['hazard_classification']['macro_f1']:.4f}")
print(f"Validation Accuracy: {metrics['hazard_classification']['accuracy']:.4f}")

print("\n=== CLINICAL CONCEPT EXTRACTION (NER) ===")
print(f"Validation Span F1: {metrics['entity_extraction_ner']['mean_span_f1']:.4f}")
print(f"Validation Precision: {metrics['entity_extraction_ner']['mean_precision']:.4f}")
print(f"Validation Recall: {metrics['entity_extraction_ner']['mean_recall']:.4f}")

## 7. Interactive End-to-End Inference Demo

In [ ]:
engine = inference.ClinicalNLPInferenceEngine()
test_note = (
    "ONCOLOGY CONSULTATION: 62yo female with Stage III NSCLC, EGFR L858R positive. "
    "Patient received cycle 2 Cisplatin 75 mg/m2. Denies chest pain or shortness of breath. "
    "Reports Grade 2 diarrhea and fatigue. Creatinine 1.2 mg/dL. ECOG PS: 1."
)
response = engine.analyze_document(test_note, document_id="DEMO-NOTE-001")
print(json.dumps(response, indent=2))